# 선수 → 팀 → 시리즈: A·B와 상체·하체 C 비교

## 고정된 실험 조건

- 2025년 555세트 / 209시리즈로 학습. 2025년 마지막 33시리즈에서만 seed별 epoch를 선택하고, 전체 2025년으로 재학습한다.
- 2026년 모델과 scaler를 고정. 완료된 186시리즈, 그에 속한 497세트로 평가한다.
- 선수 ID별 최근 5세트 기록을 사용한다. 이적해도 기록은 이어지고, 기록이 없는 선수는 2025년 fitting 구간의 포지션별 중앙값으로 보충한다.
- 선수 입력 13개: 과거 승률, 15분 골드/경험치/CS 차이, 분당 킬/데스/어시스트, DPM, 피해량 점유율, 획득 골드 점유율, CSPM, 분당 시야 점수, 과거 기록 수(최대 5)/5.
- 선수별 정답 기여도가 있는 것은 아니다. 선수 ID는 기록 연결에만 사용하며, 선수마다 별도 신경망이나 ID embedding을 만들지 않는다.

## 트랙 A: 포지션별 합산

공통 선수 인코더 13→8→4 → 각 포지션 전용 4→2→1 점수 → 다섯 점수 합. 양 팀 점수 차이를 sigmoid에 넣는다. 포지션 간 상호작용을 허용하지 않는다. 213 parameters.

## 트랙 B: 포지션 간 상호작용

같은 선수 인코더 → 포지션 순서대로 벡터 5개 결합 → 20→4→1 팀 점수. 양 팀 점수 차이를 sigmoid에 넣는다. 237 parameters. 완전히 같은 크기는 아니므로 구조만 완벽히 통제한 실험은 아니다.

## 시리즈 확률

각 seed의 세트 확률을 평균하고, Bo3는 `3p²−2p³`, Bo5는 `10p³−15p⁴+6p⁵`로 변환한다. 동일한 입력을 세 번 호출해 다수결하지 않는다.

동일·독립 세트 승률 가정이다. 시리즈 예측에는 첫 세트 출전 선수와 시리즈 시작 전 기록만 사용한다. **첫 세트 명단이 발표된 시점**을 가정하며, CSV만으로 명단 발표 시각을 검증하지는 못했다. 이후 밴픽/선수 교체를 사후 입력하지 않는다. Bo3/Bo5는 완료 승수로 복원했으며 실제 예측 시 대회 형식이 알려져 있다고 가정한다.

세트별 평가는 매 세트 직전까지의 기록과 해당 세트 선수 명단을 사용한다. 시리즈 평가는 첫 세트 입력만 사용하므로, 두 정확도는 서로 다른 평가다. 시리즈 변환은 p=0.5 경계를 유지하므로 그 자체로 승자 정확도를 높이지 않는다.

2026년은 이전 실험에서 이미 본 데이터다. 이번 실행은 2025년에서 설정을 고정하지만 엄밀한 미관측 test는 아니다.


In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display
ROOT = Path('/Users/seungyunmok/Developer/LOL_ML')
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
RESULTS = ROOT / 'notebook/experiments/09_player_two_track'

## 전체 실험 재실행

아래를 True로 바꾸면 학습과 평가를 다시 실행한다. 기본값은 저장된 결과를 확인한다. 구현은 `backend/training/player_two_track.py`에 있다.

In [2]:
RUN_EXPERIMENT = False
if RUN_EXPERIMENT:
    from backend.training.player_two_track import run
    run()

## 2025년 학습 설정

In [3]:
print(json.dumps(json.loads((RESULTS / 'lock.json').read_text()), indent=2))
display(pd.read_csv(RESULTS / 'validation_2025.csv').round(4))

{
  "epochs": {
    "position_additive": {
      "42": 8,
      "43": 7,
      "44": 77
    },
    "cross_position": {
      "42": 56,
      "43": 16,
      "44": 42
    }
  },
  "train_sets": 555,
  "train_series": 209,
  "fit_sets": 447,
  "validation_series": 33,
  "parameter_counts": {
    "position_additive": 213,
    "cross_position": 237
  },
  "primary_comparison": "2026 series accuracy; log loss secondary. Both tracks locked before reading 2026."
}


,track,n,accuracy,roc_auc,log_loss,brier_score,probability_min,probability_max
0,position_additive,33,0.5152,0.4304,0.6848,0.2459,0.3886,0.5569
1,cross_position,33,0.4545,0.4261,0.6865,0.2467,0.4122,0.5605


## 2026년 전체 결과

In [4]:
evaluation = pd.read_csv(RESULTS / 'evaluation_2026.csv')
display(evaluation.round(4))

,track,level,n,correct,accuracy,roc_auc,log_loss,brier_score,probability_min,probability_max
0,position_additive,set,497,294,0.5915,0.6206,0.6829,0.2449,0.4366,0.5703
1,position_additive,series,186,118,0.6344,0.6484,0.6754,0.2412,0.4166,0.6090
2,cross_position,set,497,277,0.5573,0.6203,0.6831,0.2450,0.4409,0.5616
3,cross_position,series,186,106,0.5699,0.6438,0.6755,0.2412,0.4193,0.5921
4,always_50,series,186,92,0.4946,0.5000,0.6931,0.2500,0.5000,0.5000
5,2025_set_winrate,series,186,94,0.5054,0.4712,0.6939,0.2504,0.4747,0.4797


## 월별 성능 및 초기화 민감도

표의 메인 성능은 세 seed의 예측 확률 평균이다. 개별 seed 중 가장 좋은 것만 골라 보고하지 않는다.

In [5]:
display(pd.read_csv(RESULTS / 'monthly_2026.csv').round(4))
display(pd.read_csv(RESULTS / 'seed_scores_2026.csv').round(4))

,track,month,n,accuracy,roc_auc,log_loss,brier_score,probability_min,probability_max
0,cross_position,2026-01,24,0.7083,0.7556,0.6616,0.2343,0.4280,0.5604
1,cross_position,2026-02,15,0.4000,0.3409,0.7086,0.2577,0.4310,0.5502
2,cross_position,2026-03,1,1.0000,NaN,0.6622,0.2345,0.4843,0.4843
3,cross_position,2026-04,44,0.6364,0.7115,0.6633,0.2352,0.4275,0.5790
4,cross_position,2026-05,46,0.7174,0.7673,0.6612,0.2341,0.4193,0.5736
5,cross_position,2026-06,5,0.4000,0.6667,0.6872,0.2473,0.4657,0.5921
6,cross_position,2026-07,6,0.3333,0.3333,0.7049,0.2559,0.4716,0.5219
7,cross_position,2026-08,38,0.3947,0.4754,0.6921,0.2495,0.4428,0.5672
8,cross_position,2026-09,7,0.2857,0.4000,0.7016,0.2543,0.4599,0.5637
9,position_additive,2026-01,24,0.7500,0.7704,0.6592,0.2331,0.4423,0.5752


,track,seed,accuracy,roc_auc,log_loss,brier_score,probability_min,probability_max
0,position_additive,42,0.5269,0.5650,0.6898,0.2483,0.4611,0.5457
1,position_additive,43,0.5914,0.6244,0.6841,0.2455,0.4397,0.5729
2,position_additive,44,0.6022,0.6393,0.6640,0.2357,0.2871,0.7358
3,cross_position,42,0.5914,0.6551,0.6632,0.2353,0.3560,0.6913
4,cross_position,43,0.5914,0.6180,0.6848,0.2459,0.4514,0.5544
5,cross_position,44,0.5753,0.6248,0.6814,0.2441,0.4349,0.5648


## 해석

포지션별 합산은 시리즈 118/186 = 63.44%, 상호작용 모델은 106/186 = 56.99%. 시리즈 log loss는 각각 0.6754와 0.6755로 거의 같다. 첫 구조가 정확도에서는 앞섰지만 확률 예측 전반이 크게 개선됐다는 증거는 아니다.

이전 팀 통계 MLP는 동일한 2026년 186시리즈에서 109/186 = 58.60%였다. 합산 모델은 9개 더 맞혔으나 학습 대상도 세트로 바뀌었으므로 개선 원인을 선수 피처 하나로만 단정하지 않는다. 8월 합산 모델 정확도는 47.37%로, 시즌 후반 안정성 문제가 남았다.

이전 같은 입력 logistic regression의 log loss 0.6661은 여전히 이번 두 모델보다 낮다. 목표가 승자 정확도인지 확률의 품질인지 구분한다. 배포 파일은 이 실험에서 변경하지 않았다.


In [6]:
print(json.dumps(json.loads((RESULTS / 'audit.json').read_text()), indent=2))
display(pd.read_csv(RESULTS / 'predictions_2026.csv').head())

{
  "cold_start_player_slots": 17,
  "series_player_slots": 1860,
  "checks": [
    "2025 inputs identical when 2026 appended",
    "team swap probabilities sum to one",
    "frozen checkpoint unchanged",
    "portable checkpoint serialization"
  ],
  "first_set_roster_assumption": true,
  "checkpoint_sha256": "bf271723e61cacde0bc54c947d4f29b3149d597cf2663f7b3f72d369921e5173",
  "source_sha256": {
    "2025": "c9a158b9e0a965a47d31d3674c127a26f75e6c91a324bd1858e4784b1336214a",
    "2026": "024330eb7a03e07c1aba55e17abf2e55f8e79e46e42e3f47389194173e59730a"
  }
}


,track,series_key,date,team1,team2,best_of,y,set_probability,series_probability
0,position_additive,2026|Cup|2026-01-14|DN SOOPers|KT Rolster,2026-01-14 08:13:43,DN SOOPers,KT Rolster,3,0,0.499383,0.499074
1,position_additive,2026|Cup|2026-01-14|Dplus Kia|HANJIN BRION,2026-01-14 11:22:18,Dplus Kia,HANJIN BRION,3,1,0.511009,0.516511
2,position_additive,2026|Cup|2026-01-15|Gen.G|Kiwoom DRX,2026-01-15 08:05:23,Gen.G,Kiwoom DRX,3,1,0.550288,0.575178
3,position_additive,2026|Cup|2026-01-15|BNK FEARX|Nongshim RedForce,2026-01-15 10:15:24,BNK FEARX,Nongshim RedForce,3,1,0.500257,0.500386
4,position_additive,2026|Cup|2026-01-16|DN SOOPers|Dplus Kia,2026-01-16 08:05:40,DN SOOPers,Dplus Kia,3,0,0.478459,0.467708


## C 추가 — 상체·하체 그룹 모델

같은 13→8→4 선수 인코더를 유지한다. 포지션 순서는 top, jng, mid, bot, sup다.

- 상체: 탑·정글·미드의 4차원 표현을 합쳐 12→2 ReLU.
- 하체: 원딜·서포터·정글의 같은 표현을 합쳐 12→2 ReLU.
- 두 그룹 표현을 결합해 4→4→1 팀 점수를 만든다. 양 팀 점수 차이가 세트 logit이다.
- 정글은 같은 벡터를 두 경로에서 재사용한다. 정글 점수를 따로 두 번 더하지 않지만 두 경로로 영향을 줄 수 있다.
- C 225개 파라미터, A 213개, B 237개. 완벽하게 같은 수는 아니지만 유사한 크기다.

데이터, 분리, optimizer, seed, early stopping 기준, 시리즈 확률 변환을 유지했다. 세 모델을 모두 재실행했으며 A·B는 이전 결과가 재현됐다. 세 모델 가중치는 2025년에서만 학습했다. 2026년은 이미 봤던 회고 평가 구간이며 구조 제안에도 이전 결과가 영향을 주었으므로 미관측 test로 주장하지 않는다.


In [7]:
GROUP_RESULTS = ROOT / 'notebook/experiments/10_player_grouped'
RUN_GROUP_EXPERIMENT = False
if RUN_GROUP_EXPERIMENT:
    from backend.training.player_two_track import run
    run(tracks=['position_additive','cross_position','upper_lower'], output_dir=GROUP_RESULTS)

group_scores = pd.read_csv(GROUP_RESULTS / 'evaluation_2026.csv')
display(group_scores.round(4))
print(json.dumps(json.loads((GROUP_RESULTS / 'lock.json').read_text()), indent=2))

,track,level,n,correct,accuracy,roc_auc,log_loss,brier_score,probability_min,probability_max
0,position_additive,set,497,294,0.5915,0.6206,0.6829,0.2449,0.4366,0.5703
1,position_additive,series,186,118,0.6344,0.6484,0.6754,0.2412,0.4166,0.6090
2,cross_position,set,497,277,0.5573,0.6203,0.6831,0.2450,0.4409,0.5616
3,cross_position,series,186,106,0.5699,0.6438,0.6755,0.2412,0.4193,0.5921
4,upper_lower,set,497,283,0.5694,0.6257,0.6837,0.2453,0.4446,0.5551
5,upper_lower,series,186,115,0.6183,0.6582,0.6757,0.2413,0.4188,0.5825
6,always_50,series,186,92,0.4946,0.5000,0.6931,0.2500,0.5000,0.5000
7,2025_set_winrate,series,186,94,0.5054,0.4712,0.6939,0.2504,0.4747,0.4797


{
  "epochs": {
    "position_additive": {
      "42": 8,
      "43": 7,
      "44": 77
    },
    "cross_position": {
      "42": 56,
      "43": 16,
      "44": 42
    },
    "upper_lower": {
      "42": 99,
      "43": 63,
      "44": 25
    }
  },
  "train_sets": 555,
  "train_series": 209,
  "fit_sets": 447,
  "validation_series": 33,
  "parameter_counts": {
    "position_additive": 213,
    "cross_position": 237,
    "upper_lower": 225
  },
  "primary_comparison": "2026 series accuracy; log loss secondary. All tracks locked before reading 2026."
}


### 결과 해석

C의 시리즈 정확도는 115/186 = 61.83%. A(118개)보다 3개 적고 B(106개)보다 9개 많다. C의 ROC-AUC는 0.6582로 세 모델 중 가장 높지만, log loss는 0.6757로 A 0.6754 / B 0.6755와 거의 같다. 정확도를 주 기준으로 정했으므로 A가 현재 기준 모델이다. A와 C의 작은 차이를 확정적 우위로 해석하지 않는다.

C의 세트 정확도는 283/497 = 56.94%. 시리즈 평가와는 관측 시점과 대상이 다르므로 시리즈 변환으로 정확도가 올랐다고 해석하지 않는다. 모델 변경 효과를 보고 추가 튜닝하지 않았다. 배포는 변경하지 않았다.


In [8]:
display(pd.read_csv(GROUP_RESULTS / 'monthly_2026.csv').round(4))
display(pd.read_csv(GROUP_RESULTS / 'seed_scores_2026.csv').round(4))

,track,month,n,accuracy,roc_auc,log_loss,brier_score,probability_min,probability_max
0,cross_position,2026-01,24,0.7083,0.7556,0.6616,0.2343,0.4280,0.5604
1,cross_position,2026-02,15,0.4000,0.3409,0.7086,0.2577,0.4310,0.5502
2,cross_position,2026-03,1,1.0000,NaN,0.6622,0.2345,0.4843,0.4843
3,cross_position,2026-04,44,0.6364,0.7115,0.6633,0.2352,0.4275,0.5790
4,cross_position,2026-05,46,0.7174,0.7673,0.6612,0.2341,0.4193,0.5736
5,cross_position,2026-06,5,0.4000,0.6667,0.6872,0.2473,0.4657,0.5921
6,cross_position,2026-07,6,0.3333,0.3333,0.7049,0.2559,0.4716,0.5219
7,cross_position,2026-08,38,0.3947,0.4754,0.6921,0.2495,0.4428,0.5672
8,cross_position,2026-09,7,0.2857,0.4000,0.7016,0.2543,0.4599,0.5637
9,position_additive,2026-01,24,0.7500,0.7704,0.6592,0.2331,0.4423,0.5752


,track,seed,accuracy,roc_auc,log_loss,brier_score,probability_min,probability_max
0,position_additive,42,0.5269,0.5650,0.6898,0.2483,0.4611,0.5457
1,position_additive,43,0.5914,0.6244,0.6841,0.2455,0.4397,0.5729
2,position_additive,44,0.6022,0.6393,0.6640,0.2357,0.2871,0.7358
3,cross_position,42,0.5914,0.6551,0.6632,0.2353,0.3560,0.6913
4,cross_position,43,0.5914,0.6180,0.6848,0.2459,0.4514,0.5544
5,cross_position,44,0.5753,0.6248,0.6814,0.2441,0.4349,0.5648
6,upper_lower,42,0.6344,0.6620,0.6577,0.2326,0.3015,0.6967
7,upper_lower,43,0.5806,0.6356,0.6860,0.2464,0.4680,0.5403
8,upper_lower,44,0.5753,0.6029,0.6913,0.2491,0.4864,0.5131
